<a href="https://colab.research.google.com/github/bhuvibhardwaj/NanoGPT/blob/main/NanoGPT_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
!nvidia-smi

Sun Aug 16 13:38:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [16]:
# Cell 2
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/NanoGPT_checkpoints', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
# Cell 3
!rm -rf /content/NanoGPT
!git clone https://github.com/bhuvibhardwaj/NanoGPT.git
%cd /content/NanoGPT
!pip install -r requirements.txt -q
!pip uninstall -y torchao -q

Cloning into 'NanoGPT'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 15 (delta 1), reused 15 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 14.11 KiB | 380.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/NanoGPT


In [26]:
# Cell 4 — the real training run, ~60 min
!python data_prep.py --source gutenberg --limit_mb 300
!python train.py

Resolving data files: 100% 37/37 [00:00<00:00, 35399.01it/s]
corpus size: 311.61M characters
encoded: 102.64M tokens, vocab_size=50257
train.bin: 92378663 tokens, val.bin: 10264296 tokens
saved to data/
/content/NanoGPT/train.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(tcfg.dtype == "fp16"))
model params: 33.46M
iter     0 | lr 1.50e-06 | train 10.7226 | val 10.7142 | 31.8s
iter   250 | lr 3.00e-04 | train 4.3212 | val 4.3901 | 185.9s
iter   500 | lr 2.98e-04 | train 3.9044 | val 4.0144 | 339.0s
iter   750 | lr 2.94e-04 | train 3.7010 | val 3.8399 | 492.1s
iter  1000 | lr 2.88e-04 | train 3.5606 | val 3.7424 | 644.9s
iter  1250 | lr 2.79e-04 | train 3.4599 | val 3.6380 | 797.3s
iter  1500 | lr 2.68e-04 | train 3.3874 | val 3.5697 | 950.1s
iter  1750 | lr 2.55e-04 | train 3.3207 | val 3.5293 | 1103.0s
iter  2000 | lr 2.41e-04 | train 3.2733 | val 3.4812

In [27]:
# Cell 5 — save immediately, don't skip this
!cp checkpoints/best.pt /content/drive/MyDrive/NanoGPT_checkpoints/best.pt
!ls -lh /content/drive/MyDrive/NanoGPT_checkpoints/

total 129M
-rw------- 1 root root 129M Aug 16 14:57 best.pt


In [28]:
# Cell 6 — sanity check
!python generate.py --prompt "The" --max_new_tokens 200

loaded checkpoint from iter 5999, val_loss=3.2157
The first person in the world is not a gentleman of

friends; and I have some experience of his sort, though my friend, that

is the most remarkable. I shall see him at once that he has no right to

hold him.”


“Well, let him come and put him into this house,” said Mr.

Wman.


“I love him!” said Mr. Bumble. “He is not a good man; he is

a man, in the way. It is a small man; he may be a fellow, and he is

going to get anything to do again.”


“Oh, well, I like our father for us,” said Carrie, as he

continued to her companion. �


In [29]:
# Cell 7 — Path A
!python instruct_data_prep.py --limit 20000
!python sft_own_model.py
!cp checkpoints_sft/sft.pt /content/drive/MyDrive/NanoGPT_checkpoints/sft.pt
!python chat_own_model.py --instruction "What is the capital of France?"

README.md: 100% 8.20k/8.20k [00:00<00:00, 15.9MB/s]

databricks-dolly-15k.jsonl: downloading bytes:   7% 860k/13.1M [00:00<00:08, 1.39MB/s]
databricks-dolly-15k.jsonl: downloading bytes: 100% 7.63M/7.63M [00:00<00:00, 11.3MB/s,  754kB/s  ]
databricks-dolly-15k.jsonl: reconstructing file: 100% 13.1M/13.1M [00:00<00:00, 19.5MB/s, 1.29MB/s  ]
Generating train split: 100% 15011/15011 [00:00<00:00, 169640.91 examples/s]
wrote 2888417 tokens from 15011 examples to data_instruct/
loaded base model, prior val_loss=3.2157
iter     0 | sft loss 8.5139
iter   200 | sft loss 5.6672
iter   400 | sft loss 5.2625
iter   600 | sft loss 5.4027
iter   800 | sft loss 5.3754
iter  1000 | sft loss 5.2646
iter  1200 | sft loss 4.8695
iter  1400 | sft loss 5.2443
iter  1600 | sft loss 5.1214
iter  1800 | sft loss 5.0004
saved to checkpoints_sft/sft.pt
The capital of Europe is the capital of Spain and France. The capital is the capital of Europe and Africa.


In [30]:
# Cell 8 — Path B
!python finetune_pretrained.py --model gpt2 --epochs 3 --limit 3000
!python chat_lora.py --model gpt2 --instruction "What is the capital of France?"

config.json: 100% 665/665 [00:00<00:00, 2.67MB/s]
tokenizer_config.json: 100% 26.0/26.0 [00:00<00:00, 163kB/s]
vocab.json: 100% 1.04M/1.04M [00:00<00:00, 6.51MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 5.40MB/s]
tokenizer.json: 100% 1.36M/1.36M [00:00<00:00, 32.3MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:  33% 180M/548M [00:00<00:01, 283MB/s, 13.1MB/s  ]
model.safetensors: downloading bytes:  54% 296M/548M [00:01<00:01, 236MB/s, 26.1MB/s  ]
model.safetensors: downloading bytes:  86% 470M/548M [00:02<00:00, 320MB/s, 38.9MB/s  ]
model.safetensors: downloading bytes: 100% 474M/474M [00:06<00:00, 74.6MB/s, 42.6MB/s  ]
model.safetensors: reconstructing file: 100% 548M/548M [00:06<00:00, 86.3MB/s, 40.9MB/s  ]
Loading weights: 100% 148/148 [00:00<00:00, 651.75it/s]
generation_config.json: 100% 124/124 [00:00<00:00, 729kB/s]
/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2504: UserWarning: fan_in_fan_out is